# Chapter 1: Marine Modeling Philosophy & Abstractions

Welcome to the foundational notebook for Guidance, Navigation, and Control (GNC). 

When engineers design software to guide a vessel across an ocean, we cannot use a single, all-encompassing mathematical formula. The real ocean is too chaotic. Instead, we use **abstractions**—simplified models tailored to specific engineering problems.

---

## The Vocabulary of Motion: 6 Degrees of Freedom (6-DOF)

To control a vessel, we must first mathematically describe its position and velocity. Fossen standardizes this across the marine industry using the standard **SNAME (1950)** coordinate vectors:

### 1. Position and Attitude Vector ($\eta$)
This vector describes the craft's position and orientation relative to the fixed Earth (the Map / NED frame).
$$\eta = [x, y, z, \phi, \theta, \psi]^T$$

* **Translational (Position):**
  * $x$ : Surge (Position along the North axis)
  * $y$ : Sway (Position along the East axis)
  * $z$ : Heave (Position along the Down axis / Depth)
* **Rotational (Attitude):**
  * $\phi$ : Roll (Tilting side-to-side)
  * $\theta$ : Pitch (Tilting front-to-back)
  * $\psi$ : Yaw (Heading angle / Compass direction)

### 2. Linear and Angular Velocity Vector ($\nu$)
This vector describes the speeds felt by sensors inside the craft itself (the Robot / BODY frame).
$$\nu = [u, v, w, p, q, r]^T$$

* **Translational (Linear Speeds):**
  * $u$ : Surge velocity (Forward speed)
  * $v$ : Sway velocity (Sideways sliding speed)
  * $w$ : Heave velocity (Vertical dive/surface speed)
* **Rotational (Angular Rates):**
  * $p$ : Roll rate
  * $q$ : Pitch rate
  * $r$ : Yaw rate (Turning speed)

---

## The Two Pillars of Marine Modeling

Fossen highlights two primary types of models used in marine engineering, depending on what problem you are trying to solve:

1. **Seakeeping Models (Frequency-Domain):**
   * **The Concept:** Think of a ship sitting still or traveling in a straight line while massive ocean waves crash against it. The main question here is: *How much does the ship shake, heave up and down, or rock left and right?*
   * **The Practical Use:** Civil and structural marine engineers use this to ensure a hull won't snap in half or capsize in a storm.

2. **Maneuvering Models (Time-Domain):**
   * **The Concept:** Think of a boat racing through calm water, twisting and turning its rudder to follow a precise path. The main question here is: *If I turn the rudder 10 degrees, what path will the boat trace out over time?*
   * **The Practical Use:** Autopilot and GNC engineers use this to design steering systems. We intentionally ignore the high-frequency chatter of waves so our computers can focus on tracking a course line.

## When Models Break Down (Linear vs. Nonlinear Theory)

Computers love **Linear Models** because they are incredibly easy and fast to compute. A linear assumption says: *"If turning my rudder 5 degrees veers me 2 meters off course, turning it 30 degrees will veer me exactly 12 meters off course."*

In the real world, fluid dynamics are **Nonlinear**. At high speeds or tight turns, the water separates from the hull, drag forces skyrocket by the square of your speed, and the linear shortcut completely tears apart. 

Let's use the interactive simulator below to see exactly where the simple math breaks down!

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import ipywidgets as widgets
from ipywidgets import interact

def simulate_vessel(speed, rudder_angle):
    """
    Simulates a true non-linear maneuvering model vs a linearized approximation
    to show where the mathematical assumptions delaminate.
    """
    t_span = (0, 20)
    t_eval = np.linspace(0, 20, 400)
    delta = np.radians(rudder_angle)
    
    # --- 1. True Nonlinear Hydrodynamic Model ---
    def nonlinear_dynamics(t, state):
        x, y, psi = state
        u = speed 
        v = 0.1 * speed * delta - 0.05 * np.abs(delta) * speed 
        r = (0.5 * speed / 5.0) * np.sin(delta) 
        
        x_dot = u * np.cos(psi) - v * np.sin(psi)
        y_dot = u * np.sin(psi) + v * np.cos(psi)
        z_dot = r
        return [x_dot, y_dot, z_dot]

    # --- 2. Linearized Model ---
    def linear_dynamics(t, state):
        x, y, psi = state
        u = speed
        v = 0.1 * speed * delta 
        r = (0.5 * speed / 5.0) * delta 
        
        x_dot = u - v * psi
        y_dot = u * psi + v
        z_dot = r
        return [x_dot, y_dot, z_dot]

    init_state = [0.0, 0.0, 0.0]
    
    sol_nl = solve_ivp(nonlinear_dynamics, t_span, init_state, t_eval=t_eval)
    sol_lin = solve_ivp(linear_dynamics, t_span, init_state, t_eval=t_eval)
    
    # --- Plotting ---
    plt.figure(figsize=(10, 6))
    plt.plot(sol_nl.y[1], sol_nl.y[0], 'b-', label='True Physical World (Nonlinear Model)', linewidth=2)
    plt.plot(sol_lin.y[1], sol_lin.y[0], 'r--', label='Autopilot Approximation (Linear Model)', linewidth=2)
    plt.plot(0, 0, 'go', label='Start Position')
    
    plt.title(f'Vessel Trajectory: Linear vs. Nonlinear Divergence\nSpeed: {speed} knots | Rudder: {rudder_angle}°', fontsize=12)
    plt.xlabel('East Displacement (Meters)', fontsize=10)
    plt.ylabel('North Displacement (Meters)', fontsize=10)
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.axis('equal')
    plt.legend(loc='upper left')
    plt.show()

print("DRAG THE SLIDERS TO EXPLORE MODEL CONVERGENCE:")
interact(simulate_vessel, 
         speed=widgets.FloatSlider(min=2.0, max=25.0, step=1.0, value=10.0, description='Speed (U):'),
         rudder_angle=widgets.FloatSlider(min=1.0, max=45.0, step=1.0, value=5.0, description='Rudder Angle:'));

DRAG THE SLIDERS TO EXPLORE MODEL CONVERGENCE:


interactive(children=(FloatSlider(value=10.0, description='Speed (U):', max=25.0, min=2.0, step=1.0), FloatSli…

### Student Lab Observations & Core GNC Takeaways

1. **Low Speed & Gentle Steering (Convergence):**
   Set the `Speed` to 5 knots and the `Rudder Angle` to 3 degrees. Notice how the blue line and red dashed line lie almost perfectly on top of each other. At this envelope, **the linear approximation is highly accurate**. Your control equations can use simple math here safely.

2. **High Speed & Aggressive Steering (Divergence/Breakdown):**
   Now, crank the `Rudder Angle` up to 35 degrees and increase the `Speed` to 22 knots. Watch how the red line completely tears away from the true blue track line. The simple linear assumptions have **diverged**. If an autopilot relies on the linear model during an aggressive evasive maneuver, it will completely miscalculate where the ship actually ends up.

3. **Why This Matters for SeaPath:**
   This is why the SeaPath navigation computer uses Fossen's rigorous kinematic matrices. We must account for the cross-coupling forces—how turning the rudder sideways physically bleeds off your forward surge energy—so our state estimator remains bulletproof across all speeds and maneuvers.